# 01 · Secure Ingestion & OCR

PDF/TXT/DOCX/CSV/XLSX/JSON/image extraction, OCR fallback for scanned PDFs, safe ZIP imports, source metadata, and page-preserving chunking. Imports foundation notebook.

In [ ]:
%run 00_foundation.ipynb

In [ ]:
import subprocess, shutil
from pypdf import PdfReader
from docx import Document as DocxDocument
from openpyxl import load_workbook
from PIL import Image

ALLOWED = {'.pdf','.txt','.md','.docx','.csv','.xlsx','.json','.png','.jpg','.jpeg'}
MIMES = {'.pdf':'application/pdf','.docx':'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
         '.png':'image/png','.jpg':'image/jpeg','.jpeg':'image/jpeg','.xlsx':'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'}

def validate_blob(filename: str, content: bytes):
    if not filename or filename != Path(filename).name or len(filename)>180 or '\\' in filename:
        raise ValueError('Unsafe or invalid filename')
    ext=Path(filename).suffix.lower()
    if ext not in ALLOWED: raise ValueError(f'Unsupported file format: {ext}')
    if not content or len(content)>MAX_UPLOAD: raise ValueError('Empty or oversized document')
    signatures={'.pdf':b'%PDF-', '.docx':b'PK\x03\x04','.xlsx':b'PK\x03\x04',
                '.png':b'\x89PNG\r\n\x1a\n','.jpg':b'\xff\xd8','.jpeg':b'\xff\xd8'}
    if ext in signatures and not content.startswith(signatures[ext]):
        raise ValueError('File signature does not match declared extension')
    return ext

def _ocr_pil(image):
    if not shutil.which('tesseract'):
        raise RuntimeError('Tesseract binary not installed; install it to OCR scanned images')
    import pytesseract
    image.thumbnail((3200,3200))
    return pytesseract.image_to_string(image)

def extract_pages(filename: str, blob: bytes):
    ext = validate_blob(filename, blob)
    if ext in {'.txt','.md'}:
        return [{'page':1,'text':blob.decode('utf-8-sig'),'section':None}]
    if ext=='.pdf':
        # PyMuPDF is much faster for 300+-page FAO reports; preserve true PDF page numbers.
        import fitz
        pages=[]
        try:
            with fitz.open(stream=blob,filetype='pdf') as pdf:
                if len(pdf)>650:raise ValueError('PDF page limit is 650')
                for i,page in enumerate(pdf,1):
                    txt=page.get_text('text',sort=True) or ''
                    if len(txt.strip())<15 and shutil.which('tesseract'):
                        try:
                            pix=page.get_pixmap(matrix=fitz.Matrix(1.8,1.8),alpha=False)
                            txt=_ocr_pil(Image.open(io.BytesIO(pix.tobytes('png'))))
                        except Exception:
                            pass  # A partially scanned page should not discard an entire long report.
                    pages.append({'page':i,'text':txt,'section':None})
        except Exception as exc:
            raise ValueError(f'PDF extraction failed for {filename}: {exc}') from exc
        return pages
    if ext=='.docx':
        d=DocxDocument(io.BytesIO(blob))
        paragraphs=[p.text for p in d.paragraphs if p.text.strip()]
        rows=[' | '.join(cell.text for cell in r.cells) for t in d.tables for r in t.rows]
        return [{'page':1,'text':'\n'.join(paragraphs+rows),'section':None}]
    if ext=='.csv':
        raw=blob.decode('utf-8-sig')
        records=list(csv.reader(io.StringIO(raw)))
        if len(records)>25000: raise ValueError('CSV row limit exceeded')
        return [{'page':1,'text':'\n'.join(' | '.join(row) for row in records),'section':'CSV'}]
    if ext=='.json':
        obj=json.loads(blob.decode('utf-8-sig'))
        return [{'page':1,'text':json.dumps(obj,indent=2,ensure_ascii=False),'section':'JSON'}]
    if ext=='.xlsx':
        book=load_workbook(io.BytesIO(blob),read_only=True,data_only=True)
        pages=[]
        try:
            if len(book.worksheets)>30: raise ValueError('Workbook worksheet limit exceeded')
            for sheet in book.worksheets:
                rows=[]
                for idx,row in enumerate(sheet.iter_rows(values_only=True)):
                    if idx>=25000: raise ValueError('XLSX row limit exceeded')
                    rows.append(' | '.join('' if x is None else str(x) for x in row))
                pages.append({'page':len(pages)+1,'text':'\n'.join(rows),'section':sheet.title})
        finally:book.close()
        return pages
    with Image.open(io.BytesIO(blob)) as img:
        img.verify()
    with Image.open(io.BytesIO(blob)) as img:
        return [{'page':1,'text':_ocr_pil(img.convert('RGB')),'section':'OCR image'}]

def make_chunks(text, limit=1000, overlap=130):
    if limit<=overlap or overlap<0: raise ValueError('Invalid chunk parameters')
    text=' '.join(text.split())
    if not text:return []
    result=[]; pos=0
    while pos<len(text):
        end=min(len(text),pos+limit)
        if end<len(text):
            break_at=text.rfind(' ',pos+int(limit*.65),end)
            if break_at>pos:end=break_at
        result.append(text[pos:end])
        if end>=len(text):break
        pos=max(pos+1,end-overlap)
    return result

def ingest_blob(project_id,domain,filename,blob,source_type='uploaded',source_url=None):
    project_id=ensure_project_id(project_id)
    if domain not in DOMAINS: raise ValueError('Invalid domain')
    ext=validate_blob(filename,blob)
    digest=hashlib.sha256(blob).hexdigest()
    with db_connect() as cx:
        prev=cx.execute('SELECT id FROM documents WHERE project_id=? AND domain=? AND sha256=?',
                        (project_id,domain,digest)).fetchone()
        if prev:return {'document_id':prev['id'],'duplicate':True,'status':'ready'}
    pages=extract_pages(filename,blob)
    parts=[(p['page'],p['section'],ch) for p in pages for ch in make_chunks(p['text'])]
    if not parts:raise ValueError('No text extracted; cannot create searchable evidence')
    dest=STORE/'uploads'/project_id
    dest.mkdir(parents=True,exist_ok=True)
    name=digest+ext
    tmp=dest/(name+'.tmp')
    tmp.write_bytes(blob)
    os.replace(tmp,dest/name)
    doc_id=str(uuid.uuid4())
    with db_connect() as cx:
        cx.execute('INSERT INTO documents VALUES (?,?,?,?,?,?,?,?,?,?,?,?)',
                   (doc_id,project_id,domain,filename,digest,str((dest/name).relative_to(ROOT)),
                    MIMES.get(ext,mimetypes.guess_type(filename)[0] or 'application/octet-stream'),
                    source_type,source_url,len(blob),'ready',NOW()))
        for idx,(page,section,text) in enumerate(parts):
            cx.execute('INSERT INTO chunks VALUES (?,?,?,?,?,?,?,?)',
                       (str(uuid.uuid4()),doc_id,project_id,domain,page,idx,text,json.dumps({'section':section})))
    return {'document_id':doc_id,'chunks':len(parts),'status':'ready','duplicate':False,'domain':domain}

def ingest_file(path,domain,project_id=PROJECT_ID,source_type='course-provided'):
    path=Path(path)
    return ingest_blob(project_id,domain,path.name,path.read_bytes(),source_type=source_type)

def ingest_archive(zip_path,domain,project_id=PROJECT_ID):
    outputs=[]; cumulative=0
    with zipfile.ZipFile(zip_path) as archive:
        entries=[x for x in archive.infolist() if not x.is_dir()]
        if len(entries)>150:raise ValueError('ZIP file count exceeds 150')
        for member in entries:
            name=member.filename
            pp=PurePosixPath(name)
            if name.startswith('/') or '\\' in name or '..' in pp.parts or member.file_size>MAX_UPLOAD:
                raise ValueError('Unsafe archive path or member size')
            cumulative+=member.file_size
            if cumulative>100*1024*1024:raise ValueError('ZIP total uncompressed size exceeded')
            safe_name=pp.name
            if Path(safe_name).suffix.lower() not in ALLOWED or safe_name.startswith('.'):continue
            try:outputs.append({'file':name,**ingest_blob(project_id,domain,safe_name,archive.read(member),source_type='course-archive')})
            except Exception as exc:outputs.append({'file':name,'error':str(exc)})
    return outputs
print('Document ingestion functions ready')